# STAR-v3 — Kaggle V2: EXPERIMENT RUNNER (10 epoch, sweep LR, LHP off, λ₂=0.5)

**Cai dat:** Accelerator = **GPU T4** · Internet = **ON** · Add Input dataset nhu v1 (tar.zst + xvlm_16m_base.th).

**Khac v1:** chay NHIEU thi nghiem tuan tu trong 1 session, tu tong hop bang so sanh cuoi.
Cau hinh v2 mac dinh (theo yeu cau): **LHP OFF** · pose ON · **λ₂ = 0.5** · **10 epoch** + patience 6 lan eval
(= 3 epoch khong cai thien moi dung — tranh dung som o cuc tri dia phuong nhu v1 dung @2.5 epoch) ·
**batch 20** (tang tu 16; gate cell se kiem OOM truoc khi chay dai, log co `vram=`) · sweep **LR {2e-4, 1e-4}**.

**Tra loi nhanh 2 cau hoi:**
- **fp32 co tot len khong?** Tren T4: KHONG nen — cham 3–8× (tensor core T4 chi manh fp16) va ton gap doi
  VRAM → phai ha batch → it negative ITC hon → thuong TE hon. fp16+GradScaler la che do chuan cua dong model
  nay; da co grad-clip + NaN-guard. (Co dong thi nghiem fp32 comment san neu van muon tu kiem chung.)
- **Tang batch co tot hon khong?** Co kha nang CO (batch that = so negative ITC moi step). v1 batch 16 chua OOM;
  v2 mac dinh 20, gate cell chay truoc 5 phut o dung batch do — neu OOM thi biet ngay, ha ve 16. Log in `vram=`.

**Thoi gian uoc tinh:** moi run 10-epoch toi da ~2h (thuong dung som hon do patience); 2 run mac dinh + gate ≈ **3.5–4.5h**.

In [ ]:
# [1/8] GPU check + clone/pull repo
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "KHONG THAY GPU — bat Settings > Accelerator = GPU T4 roi chay lai!"
print("GPU:", gpu)
os.chdir("/kaggle/working")
if pathlib.Path("star/.git").exists():
    !cd star && git pull -q
else:
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
assert pathlib.Path("src/star/data/dataset.py").exists(), "repo thieu src/star/data — git pull lai!"
print("repo OK:", os.getcwd())

In [ ]:
# [2/8] Env + X-VLM + 4 patches (logic trong repo, idempotent)
!python scripts/kaggle_setup.py

In [ ]:
# [3/8] Tim dataset: checkpoint + data (tar.zst hoac folder)
import glob, os, pathlib, shutil

def find_one(pattern):
    hits = sorted(set(glob.glob(f"/kaggle/input/*/{pattern}") +
                      glob.glob(f"/kaggle/input/*/**/{pattern}", recursive=True)))
    return hits[0] if hits else None

CKPT  = find_one("xvlm_16m_base.th")
JSONL = find_one("train_10k_hard.jsonl")
ARCH  = find_one("train_10k_hard_data.tar.zst")

if JSONL:
    DATA_ROOT = str(pathlib.Path(JSONL).parents[2])
elif ARCH:
    marker = "/kaggle/working/extract"
    got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
    if not got:
        os.makedirs(marker, exist_ok=True)
        print("extracting 1.6GB (~2-4 phut)...")
        if shutil.which("zstd"):
            os.system(f"tar -I 'zstd -d' -xf '{ARCH}' -C {marker}")
        else:
            import zstandard, tarfile
            with open(ARCH, "rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as zr:
                with tarfile.open(fileobj=zr, mode="r|") as tf:
                    tf.extractall(marker)
        got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
    assert got, "giai nen that bai"
    DATA_ROOT = str(pathlib.Path(got[0]).parents[2])
else:
    raise FileNotFoundError("Khong thay data! Add Input dataset truoc.")

if not CKPT:
    os.makedirs("data/checkpoints", exist_ok=True)
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
    CKPT = "data/checkpoints/xvlm_16m_base.th"
assert pathlib.Path(CKPT).stat().st_size > 8e8
print("DATA_ROOT =", DATA_ROOT)
print("CKPT      =", CKPT)

In [ ]:
# [4/8] Build manifest + ViTPose enrich (y het v1 — da kiem: 11,735 rows, coverage 100%, leakage 0)
import json, pathlib
import numpy as np, pandas as pd

root = pathlib.Path(DATA_ROOT)
anchors, hard_rows, anchor_ids = [], {}, set()
def bucket_of(p):
    return "wentwrong" if "/wentwrong/" in p else ("full" if "/full/" in p else "goal")

for line in open(root / "data/subsets/train_10k_hard.jsonl", encoding="utf-8"):
    r = json.loads(line)
    anchor_ids.add(r["image_id"])
    anchors.append(dict(image_path=r["image_webp"], caption=r["caption"],
        sequence_id=f'v{r["video_id"]}_{r["bucket"]}', scene=f'v{r["video_id"]}',
        action=str(r.get("label", "unk")), video_id=r["video_id"], image_id=r["image_id"]))
    hid = r.get("hard_i_id")
    if hid and hid not in hard_rows:
        hard_rows[hid] = dict(image_path=r["hard_image_webp"], caption=r["hard_c"],
            sequence_id=f'v{r["video_id"]}_{bucket_of(r["hard_image_webp"])}', scene=f'v{r["video_id"]}',
            action="hard_pair", video_id=r["video_id"], image_id=hid)

df = pd.DataFrame(anchors + [v for k, v in hard_rows.items() if k not in anchor_ids])
pose = json.load(open(root / "data/subsets/prepared/train_10k_hard_vitpose.json", encoding="utf-8"))["items"]
def kpts_of(iid):
    it = pose.get(iid)
    if not it or it.get("status") != "ok" or not it.get("instances"):
        return None
    W, H = it.get("width", 384), it.get("height", 384)
    flat = []
    for x, y, c in it["instances"][0]["keypoints_xyc"]:
        flat += [x / W, y / H, c]
    return flat if len(flat) == 51 else None
def bbox_of(iid):
    it = pose.get(iid)
    b = it.get("primary_bbox_norm_xyxy") if it else None
    if not b:
        return None
    x1, y1, x2, y2 = b
    return [x1, y1, max(x2 - x1, 1e-3), max(y2 - y1, 1e-3)]
df["keypoints"] = df.image_id.map(kpts_of)
df["bbox"] = df.image_id.map(bbox_of)

rng = np.random.default_rng(42)
vids = df.video_id.unique().copy(); rng.shuffle(vids)
counts = df.groupby("video_id").size(); vq, vd, acc = set(), set(), 0
it = iter(vids)
for v in it:
    vq.add(v); acc += counts[v]
    if acc >= 600: break
acc = 0
for v in it:
    vd.add(v); acc += counts[v]
    if acc >= 900: break
df["split"] = np.where(df.video_id.isin(vq | vd), "valb", "train")
df.loc[df.video_id.isin(vd), "caption"] = ""

MANIFEST = "/kaggle/working/manifest_10k_hard.parquet"
df.to_parquet(MANIFEST, index=False)
leak = len(set(df[df.split == "train"].video_id) & set(df[df.split == "valb"].video_id))
print(f"rows={len(df)}  train={(df.split=='train').sum()}  "
      f"valb_q={((df.split=='valb') & (df.caption!='')).sum()}  valb_d={((df.split=='valb') & (df.caption=='')).sum()}")
print(f"kpts={df.keypoints.notna().mean():.0%} bbox={df.bbox.notna().mean():.0%} leakage={leak}")
assert leak == 0 and df.keypoints.notna().mean() > 0.99

In [ ]:
# [5/8] ====== BANG DIEU KHIEN THI NGHIEM (EDIT O DAY) ======
DATA = f"data.manifest={MANIFEST} data.image_root={DATA_ROOT} model.checkpoint={CKPT}"

# Cau hinh CHUNG cua v2: LHP OFF, pose ON, 10 epoch, patience 6 eval (=3 epoch khong cai thien),
# batch 20 (tang tu 16 — gate cell kiem OOM truoc khi chay dai)
COMMON = ("data.lhp_enabled=false model.pose_enabled=true "
          "optim.epochs=10 train.early_stop_patience=6 train.eval_every_epochs=0.5 "
          "train.batch_size=20 train.grad_accum=2")

EXPERIMENTS = [
    {"name": "v2a_smap05_lr2e-4", "set": "loss.lambda_smooth_ap=0.5 optim.lr_lora=2e-4"},
    {"name": "v2b_smap05_lr1e-4", "set": "loss.lambda_smooth_ap=0.5 optim.lr_lora=1e-4"},
    # --- mo them neu con gio (bo comment) ---
    # {"name": "v2c_smap0_doi_chung",  "set": "loss.lambda_smooth_ap=0 optim.lr_lora=2e-4"},
    # {"name": "v2d_lr5e-4",           "set": "loss.lambda_smooth_ap=0.5 optim.lr_lora=5e-4"},
    # {"name": "v2e_batch24",          "set": "loss.lambda_smooth_ap=0.5 optim.lr_lora=2e-4 train.batch_size=24"},
    # {"name": "v2f_warmup05",         "set": "loss.lambda_smooth_ap=0.5 optim.lr_lora=2e-4 optim.warmup_epochs=0.5"},
    # KHONG khuyen nghi (cham 3-8x, de OOM, thuong te hon — chi de tu kiem chung):
    # {"name": "v2x_fp32_batch8",      "set": "loss.lambda_smooth_ap=0.5 optim.lr_lora=2e-4 train.amp_dtype=fp32 train.batch_size=8"},
]
print(f"{len(EXPERIMENTS)} thi nghiem; COMMON: {COMMON}")

In [ ]:
# [6/8] GATE: overfit-one-batch o DUNG batch cua v2 — vua kiem wiring vua kiem OOM (xem dong vram=)
gate_cmd = (f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --overfit-one-batch "
            f"--set {DATA} {COMMON} {EXPERIMENTS[0]['set']}")
print(gate_cmd)
!{gate_cmd}
print("Neu OOM: ha train.batch_size trong COMMON ve 16 roi chay lai cell nay.")

In [ ]:
# [7/8] CHAY TUAN TU CAC THI NGHIEM — moi run luu rieng best.pth, gom ket qua dan
import pathlib, time, torch
RESULTS = []
for exp in EXPERIMENTS:
    out_dir = f"/kaggle/working/{exp['name']}"
    cmd = (f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml "
           f"--set {DATA} {COMMON} {exp['set']} train.out_dir={out_dir}")
    print("=" * 100); print(">>>", exp["name"]); print(cmd)
    t0 = time.time()
    !{cmd}
    mins = round((time.time() - t0) / 60, 1)
    best, report = None, None
    ck = pathlib.Path(out_dir, "best.pth")
    if ck.exists():
        raw = torch.load(ck, map_location="cpu", weights_only=False)
        best = raw.get("best_metric")
        report = (raw.get("extra") or {}).get("report")
        del raw
    RESULTS.append(dict(name=exp["name"], mAP=best, mins=mins, report=report))
    print(f"<<< {exp['name']}: best mAP = {best}  ({mins} phut)")

In [ ]:
# [8/8] BANG SO SANH CUOI (moc v1: mAP 0.6615 — LHP on, pose on, smap 0.2, lr 2e-4, batch 16, dung @2.5ep)
import pandas as pd
tab = pd.DataFrame([{ "exp": r["name"], "best_mAP": r["mAP"], "phut": r["mins"],
                      "R@1": (r["report"] or {}).get("R@1"), "R@5": (r["report"] or {}).get("R@5"),
                      "R@10": (r["report"] or {}).get("R@10") } for r in RESULTS])
tab = tab.sort_values("best_mAP", ascending=False).reset_index(drop=True)
print(tab.to_string())
print("\nMoc v1 de so: mAP=0.6615 R@1=0.5298 (chenh < ~0.01 mAP voi 621 query = NHIEU, coi nhu hoa)")
!ls -lh /kaggle/working/*/best.pth 2>/dev/null

## Doc ket qua cho dung
- **So voi moc v1 (0.6615):** chenh lech **< ~0.01 mAP coi nhu HOA** (VAL-B chi 621 query — nhieu seed/sampling). Chi ket luan khi chenh ro rang.
- v2 doi nhieu bien cung luc so voi v1 (LHP off, λ₂ 0.2→0.5, batch 16→20, lich 10ep): neu v2a thang v1, muon biet **nho cai gi** thi chay them doi chung (v2c: λ₂=0; hoac bat lai LHP).
- **Tai checkpoint tot nhat** tu Output: `/kaggle/working/<ten_exp>/best.pth` → ban giao theo `reports/instruct_inference.md`.
- LUU Y pose van ON trong moi exp v2 → inference van can ViTPose keypoints cho gallery.

## Ghi chu ky thuat
- **fp32:** khong khuyen nghi tren T4 (cham 3–8×, gap doi VRAM → batch nho hon → it negative ITC → thuong te hon). fp16 + GradScaler + grad-clip + NaN-guard la du an toan.
- **Batch:** dong `vram=` trong log cho biet peak; batch 20 fp16 du kien ~12–15G/15.36G. Neu OOM o gate → ha 16. Muon thu 24 → mo exp v2e.
- **10 epoch + patience 6:** cosine LR gian ra theo epochs=10 → LR cuoi ky thap hon han v1 → co co hoi vuot plateau 0.656–0.658 cua v1. Khong dam bao — do moi biet.